# **Exercise 1 — Setup & Environment**

In [ ]:
!pip install -qU langchain langchain-community langgraph "langchain[mistralai]" tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==

In [ ]:
# Enabling LangSmith tracing
import os, getpass

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("🔑 LangSmith API Key: ")
print("LangSmith tracing on =", os.getenv("LANGSMITH_TRACING") == "true")

🔑 LangSmith API Key: ··········
LangSmith tracing on = True


In [ ]:
# Tavily API Key
import os, getpass
os.environ["TAVILY_API_KEY"] = getpass.getpass("🔑 Tavily API Key: ")
print("Tavily key set =", os.getenv("TAVILY_API_KEY") is not None)

🔑 Tavily API Key: ··········
Tavily key set = True


# **Exercise 2 — Define Tools (Tavily Search)**

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults()         # reads TAVILY_API_KEY from env
res = search.run("latest advancements in AI-driven healthcare triage")
print(type(res), "\n", res[:300], "…")
tools = [search]

ModuleNotFoundError: No module named 'langchain_community'

# **Exercise 3 — Using Language Models (Mistral via LangChain)**

In [ ]:
!pip install -qU "langchain[mistralai]" langchain-core langchain

In [ ]:
import os, getpassimport os, getpass

# Re-set the env var cleanly (no emojis, strip whitespace/newlines)
if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API Key: ").strip()

print("Key length:", len(os.environ["MISTRAL_API_KEY"]))
assert len(os.environ["MISTRAL_API_KEY"]) > 10, "API key looks empty."

# Re-set the env var cleanly (no emojis, strip whitespace/newlines)
if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API Key: ").strip()

print("Key length:", len(os.environ["MISTRAL_API_KEY"]))
assert len(os.environ["MISTRAL_API_KEY"]) > 10, "API key looks empty."

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage

model = ChatMistralAI(
    model="mistral-small-latest",   # choose ONE model
    temperature=0.7,
    api_key=os.environ["MISTRAL_API_KEY"]
)

resp = model([HumanMessage(content="Tell me a fun fact about space exploration.")])
print(resp.content)

# **Exercise 4 — Bind Tools & Inspect Responses**

In [ ]:
model_with_tools = model.bind_tools(tools)

# Case A: no tool needed
resp_a = model_with_tools([HumanMessage(content="Tell me a robot joke.")])
print("A.content:", resp_a.content)
print("A.tool_calls:", resp_a.tool_calls)  # likely []

# Case B: tool likely required
resp_b = model_with_tools([HumanMessage(
    content="Search for the latest AI breakthroughs in healthcare."
)])
print("B.content:", resp_b.content)        # often empty if it plans a tool call
print("B.tool_calls:", resp_b.tool_calls)  # should show a Tavily call + args

# **Exercise 5 — Create the Agent (LangGraph ReAct)**

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm=model, tools=tools)
agent  # just to display the object

# **Exercise 6 — Run the Agent (invoke & helper)**

In [ ]:
from langchain_core.messages import HumanMessage

def ask(prompt: str):
    state = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    # `state` is a dict-like; final text is usually in `messages`[-1].content
    last_msg = state["messages"][-1]
    print(last_msg.content)
    return state

# Stateless Q&A
s1 = ask("What is the capital of France?")
s2 = ask("Tell me a short joke about robots.")

# Query that should trigger a tool
s3 = ask("Find the latest breakthroughs in AI for emergency medicine.")
s3_tool_calls = getattr(s3["messages"][-1], "tool_calls", None)
print("Tool calls:", s3_tool_calls)

# **Exercise 7 — Streaming Agent Output**

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="Find the top 3 recent AI papers on medical imaging.")]
stream = agent.stream({"messages": messages}, stream_mode="auto")

for step in stream:
    # Each step is a partial state; the newest message is usually last
    msgs = step.get("messages", [])
    if msgs:
        m = msgs[-1]
        # `.pretty_print()` is handy; fall back to printing content
        try:
            m.pretty_print()
        except Exception:
            print(getattr(m, "content", m))